<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-10-run-the-cobalt-model-like-a-service.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 10 (graded) — Run the Cobalt model like a service
**Course 1: Hands-On Deep Learning with Python — Chapter 10: Deploy, monitor, detect drift**

**Problem brief (Leo Farkas, Cobalt Manufacturing, two weeks after Chapter 5 shipped):**
"The defect model was great for a week. Then the line supervisor swapped a camera and now
we're getting false alarms. How do we run this thing properly?"

**What you'll submit:** the Chapter 5 model served behind a prediction function, fed the
clean stream then a simulated camera-shift stream, a drift report showing the monitor
catching it, and a retraining-decision memo.

In [ ]:
!pip install -q evidently

## 1. Train (or reload) the Chapter 5 defect detector
Uses the same synthetic-image generator as Chapter 5/8's offline fallback for a fast,
self-contained lab; swap in your real Chapter 5 model + data if you trained one.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG = 32

def make_casting_like(n, defect_rate, rng, brightness_shift=0.0, contrast_scale=1.0):
    """brightness_shift/contrast_scale simulate a camera change — set them != 0/1 to
    generate the 'new camera' stream."""
    X, y = [], []
    for i in range(n):
        label = int(rng.random() < defect_rate)
        base = rng.normal(0.5, 0.08, (IMG, IMG)).astype(np.float32)
        if label == 1:
            yy, xx = np.mgrid[0:IMG, 0:IMG]
            cy, cx = rng.integers(8, IMG - 8, 2)
            r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2)
            base = np.clip(base + np.exp(-((r - 6) ** 2) / 5.0) * 0.5, 0, 1)
        base = np.clip((base - 0.5) * contrast_scale + 0.5 + brightness_shift, 0, 1)
        X.append(base); y.append(label)
    return np.array(X, dtype=np.float32), np.array(y)

rng = np.random.default_rng(0)
X_train, y_train = make_casting_like(1200, 0.2, rng)

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
    nn.Flatten(), nn.Linear(32, 2),
).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.tensor(X_train).unsqueeze(1), torch.tensor(y_train, dtype=torch.long)),
    batch_size=32, shuffle=True,
)
for epoch in range(10):
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step()
model.eval()
print('Model trained. Final batch loss:', loss.item())

## 2. Package as a service
Real deployment target: **BentoML**. If it's not installed/available in your runtime, this
cell falls back to an equivalent plain-Python service class — the `predict()` contract
(input validation, structured output, latency) is what actually matters for this lab.

In [ ]:
import time

class DefectDetectorService:
    """predict(image: np.ndarray[32,32]) -> {label, score, latency_ms} — the same contract
    Chapter 5's `predict(image) -> {label, score, latency_ms}` function used, now wrapped for
    serving. In a real BentoML deployment this class's logic lives inside a
    @bentoml.service-decorated class with the model loaded via bentoml.pytorch.get(...)."""

    def __init__(self, model):
        self.model = model
        self.log = []  # the request log every production model needs (Chapter 10 "what to log")

    def predict(self, image):
        if image.shape != (IMG, IMG):
            return {'error': f'expected a {IMG}x{IMG} image, got {image.shape}'}
        t0 = time.perf_counter()
        x = torch.tensor(image, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        with torch.no_grad():
            probs = torch.softmax(self.model(x.to(device)), dim=1)[0]
        latency_ms = (time.perf_counter() - t0) * 1000
        result = {
            'label': 'defective' if probs[1] > 0.5 else 'ok',
            'score': float(probs[1]),
            'latency_ms': round(latency_ms, 2),
        }
        self.log.append({'image_mean': float(image.mean()), 'image_std': float(image.std()), **result})
        return result

    def health(self):
        return {'status': 'ok', 'requests_served': len(self.log)}

service = DefectDetectorService(model)
print(service.health())
print(service.predict(X_train[0]))

## 3. Feed the clean stream, then the shifted (new-camera) stream

In [ ]:
clean_X, clean_y = make_casting_like(300, 0.2, np.random.default_rng(10))
for img in clean_X:
    service.predict(img)

shifted_X, shifted_y = make_casting_like(
    300, 0.2, np.random.default_rng(11), brightness_shift=0.18, contrast_scale=1.6,
)
for img in shifted_X:
    service.predict(img)

print(f'Requests served: {service.health()["requests_served"]} (300 clean + 300 shifted)')

## 4. Detect the drift
A from-scratch PSI check (guaranteed to work, matches the chapter's derivation) plus an
Evidently report if the library installed cleanly in your runtime.

In [ ]:
import pandas as pd

def psi(expected, actual, bins=10):
    edges = np.histogram(expected, bins=bins)[1]
    e_counts, _ = np.histogram(expected, bins=edges)
    a_counts, _ = np.histogram(actual, bins=edges)
    e_pct = np.clip(e_counts / len(expected), 1e-4, None)
    a_pct = np.clip(a_counts / len(actual), 1e-4, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

log_df = pd.DataFrame(service.log)
clean_log = log_df.iloc[:300]
shifted_log = log_df.iloc[300:]

psi_mean = psi(clean_log['image_mean'], shifted_log['image_mean'])
psi_std = psi(clean_log['image_std'], shifted_log['image_std'])
print(f'PSI on image brightness (mean): {psi_mean:.3f}  (>0.2 = investigate/retrain)')
print(f'PSI on image contrast (std):    {psi_std:.3f}')
print(f'False-alarm rate, clean stream:   {(clean_log["label"] == "defective").mean():.1%}')
print(f'False-alarm rate, shifted stream: {(shifted_log["label"] == "defective").mean():.1%}')

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    reference = clean_log[['image_mean', 'image_std', 'score']]
    current = shifted_log[['image_mean', 'image_std', 'score']]
    report = Report(metrics=[DataDriftPreset()])
    report.run(reference_data=reference, current_data=current)
    report.save_html('evidently_drift_report.html')
    print('\nEvidently drift report saved to evidently_drift_report.html — open it in Colab\'s file browser.')
except Exception as e:
    print(f'\nEvidently report skipped ({e}) — the PSI check above still demonstrates the same')
    print('detection idea and is what the assertion below actually checks.')

assert psi_mean > 0.2 or psi_std > 0.2, 'The monitor should have flagged this shift — check your drift simulation.'
print('\nMonitor correctly flagged the camera change.')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(clean_log['image_mean'], bins=20, alpha=0.6, label='clean')
axes[0].hist(shifted_log['image_mean'], bins=20, alpha=0.6, label='shifted')
axes[0].set_title('Brightness distribution'); axes[0].legend()
axes[1].plot(log_df['score'].values)
axes[1].axvline(300, color='r', linestyle='--', label='camera swapped here')
axes[1].set_title('Defect score over the request stream'); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Retraining-decision memo (fill in)
Based on the PSI values and the false-alarm-rate jump above: is this a retrain-now situation,
a recalibrate-the-camera situation, or something else? Who needs to approve before anything
ships (Chapter 10's human-in-the-loop point)? Sketch the retrain trigger you'd wire into an
Airflow DAG.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 10: Deploy, monitor, detect drift*